# Notebook 02: Interview Prompt Engineering — OFPHIL MSW Interview Assistant

**Project:** `ofphil-classification-v2`  
**Phase:** 05 Model Selection and Training — Track B: LLM Interview Assistant  
**Notebook:** `02_interview_prompt_engineering.ipynb`  
**Role:** Prompt Architecture, Persona Constraints, and Multi-Domain Interview Templates

---

### Objectives
1. Define the core System Prompt for the OFPHIL Medical Social Worker (MSW) Interview Assistant.
2. Establish strict behavioral guardrails:
   - Zero eligibility prediction or assistance approval/disapproval.
   - Zero severity, coping, or financial scoring (all scoring belongs to deterministic backend algorithms).
   - Zero unsupported assumptions or hallucinated patient details.
3. Construct structured interview prompt templates covering the 5 primary MSW intake domains:
   - **Domain 1: Medical Information & Clinical Circumstances** (Treatment history, diagnosis, surgical urgency).
   - **Domain 2: Socioeconomic Circumstances** (Housing status, utilities, living environment).
   - **Domain 3: Financial Circumstances** (Informal/daily income, recurring obligations, debt).
   - **Domain 4: Household and Family Circumstances** (Dependents, caregiver burdens, breadwinner status).
   - **Domain 5: Psychosocial Circumstances** (Support systems, emotional coping, crisis indicators).
4. Formulate contextual follow-up generation prompts with empathetic, Taglish-aware, respectful phrasings.
5. Export structured prompt templates and test schemas to `data/interim/llm_interview/prompts/`.


In [ ]:
from pathlib import Path
import json

def _find_project_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / 'data').exists() and (p / 'experiments').exists():
            return p
    return Path('../../..').resolve()

PROJECT_ROOT = _find_project_root(Path.cwd())
PROMPTS_DIR = PROJECT_ROOT / 'data' / 'interim' / 'llm_interview' / 'prompts'
PROMPTS_DIR.mkdir(parents=True, exist_ok=True)

SYSTEM_PROMPT_PATH = PROMPTS_DIR / 'msw_assistant_system_prompt.txt'
DOMAIN_PROMPTS_JSON = PROMPTS_DIR / 'domain_interview_prompts.json'
FOLLOWUP_PROMPTS_JSON = PROMPTS_DIR / 'followup_generation_prompts.json'

print(f"Prompts Directory: {PROMPTS_DIR}")


### 2. Core System Prompt Definition
The System Prompt enforces the clinical boundary of the assistant: it is an interview copilot for licensed Medical Social Workers, not an automated adjudicator.


In [ ]:
SYSTEM_PROMPT = """You are the OFPHIL Medical Social Service Interview Copilot, an AI assistant designed to support licensed Medical Social Workers (MSWs) during Stage 2 patient intake interviews.

ROLE AND PURPOSE:
- You assist the MSW by proposing clear, respectful, patient-friendly follow-up questions and structuring patient statements.
- You operate strictly under the direction of the human MSW. The MSW may accept, modify, or reject any suggestion.

STRICT BEHAVIORAL BOUNDARIES & GUARDRAILS:
1. NO ELIGIBILITY DECISIONS: You MUST NEVER state, imply, or predict whether a patient is 'ELIGIBLE' or 'INELIGIBLE' for charity assistance. Eligibility is adjudicated exclusively by institutional MSW guidelines and quantitative models.
2. NO ASSIGNMENT OF SCORES: You MUST NEVER calculate or assign medical severity ratings, coping scores, or financial distress indices. Deterministic calculations are performed by the backend.
3. NO HALLUCINATION OR FABRICATION: Do not invent family members, clinical conditions, income streams, or expenses. If an answer is unclear, ambiguous, or missing, explicitly flag it as 'INCOMPLETE' or 'UNCLEAR'.
4. LANGUAGE & TONE: Use respectful, empathetic, non-judgmental language. Support Tagalog, English, and natural Taglish colloquialisms (e.g., 'arawan', 'boundary', 'pamasahe', 'pang-maintenance'). Avoid medical or bureaucratic jargon when speaking to patients.
5. PRESERVE ORIGINAL VOICE: In extraction mode, preserve the verbatim meaning of patient statements. Clearly distinguish between what the patient explicitly stated versus observational casework notes recorded by the MSW.
6. CONVERSATION CONTEXT: Track previously discussed topics. Never propose a question that directly duplicates information already provided in the interview history."""

SYSTEM_PROMPT_PATH.write_text(SYSTEM_PROMPT, encoding='utf-8')
print(f"Saved Core System Prompt -> {SYSTEM_PROMPT_PATH}")


### 3. Multi-Domain Structured Interview Prompts
The intake process is broken down into 5 standard MSW assessment domains. For each domain, we define:
- Primary intake objectives
- Suggested initial exploratory questions (in English and natural conversational Taglish)
- Required observational cues for the caseworker


In [ ]:
domain_prompts = {
    "domain_1_medical_information": {
        "title": "Medical Information & Clinical Circumstances",
        "description": "Explores the patient's visual impairment diagnosis, recommended treatment/surgery, and past ophthalmologic interventions.",
        "core_questions": [
            {
                "question_id": "MED-Q01",
                "topic": "Current Visual Complaint & History",
                "prompt_en": "Can you describe the vision problems you are experiencing and how long they have been affecting you?",
                "prompt_taglish": "Maaari po ba ninyong maipaliwanag kung ano ang nararamdaman sa inyong paningin at gaano na po ito katagal?",
                "expected_entities": ["affected_eye_laterality", "symptom_onset_duration", "impact_on_daily_activities"]
            },
            {
                "question_id": "MED-Q02",
                "topic": "Recommended Eye Procedure & Quotation",
                "prompt_en": "What specific eye surgery or procedure was recommended by your ophthalmologist, and what was the quoted procedure cost?",
                "prompt_taglish": "Ano po ang eksaktong operasyon o gamutan na sinabi ng doktor para sa inyong mata, at magkano po ang tantiyang halaga?",
                "expected_entities": ["procedure_name", "procedure_cost_quoted", "facility_name"]
            }
        ]
    },
    "domain_2_socioeconomic_circumstances": {
        "title": "Socioeconomic Circumstances & Living Conditions",
        "description": "Audits living environment, housing tenure, access to basic utilities, and geographic vulnerability.",
        "core_questions": [
            {
                "question_id": "SOC-Q01",
                "topic": "Housing Ownership & Tenurial Status",
                "prompt_en": "What is the status of your current dwelling (e.g., owned, rented, living with relatives, or informal settler)?",
                "prompt_taglish": "Ano po ang estado ng inyong tinitirhan sa kasalukuyan (sarili po ba, nangungupahan, nakikitira sa kamag-anak, o informal settlement)?",
                "expected_entities": ["housing_tenure", "monthly_rent_amount", "dwelling_type"]
            },
            {
                "question_id": "SOC-Q02",
                "topic": "Basic Utilities and Water Access",
                "prompt_en": "Do you have formal electricity and running water connections, and how much do you spend monthly on utilities?",
                "prompt_taglish": "May sarili po ba kayong koneksyon sa kuryente at tubig, at magkano po karaniwan ang inyong buwanang bayarin?",
                "expected_entities": ["electricity_source", "water_source", "monthly_utility_estimate"]
            }
        ]
    },
    "domain_3_financial_circumstances": {
        "title": "Financial Circumstances, Livelihood & Debt",
        "description": "Gathers detailed household earnings, irregular livelihood cash flows, outstanding debts, and external financial assistance.",
        "core_questions": [
            {
                "question_id": "FIN-Q01",
                "topic": "Household Earnings & Livelihood Stability",
                "prompt_en": "Who contributes to the household income, what are their occupations, and how much is earned on average per week or month?",
                "prompt_taglish": "Sino-sino po ang may hanapbuhay sa inyong pamilya, ano po ang kanilang trabaho, at magkano po ang karaniwang kinikita kada linggo o buwan?",
                "expected_entities": ["working_members_occupations", "monthly_family_income_estimate", "income_regularity"]
            },
            {
                "question_id": "FIN-Q02",
                "topic": "Existing Assistance & Outstanding Hospital Bills",
                "prompt_en": "Have you received guarantee letters or cash assistance from PhilHealth, PCSO, DSWD, or local officials, and do you have existing hospital debts?",
                "prompt_taglish": "May natanggap na po ba kayong tulong tulad ng GL mula sa PhilHealth, DSWD, OVP, o Mayor, at may naiwan pa po bang bayarin sa ospital?",
                "expected_entities": ["existing_assistance_sources", "existing_assistance_amount", "outstanding_hospital_debt"]
            }
        ]
    },
    "domain_4_household_and_family": {
        "title": "Household Composition & Caregiver Dependency",
        "description": "Examines household size, vulnerable dependents (children, elderly, PWD), and single-parent or sole-breadwinner burdens.",
        "core_questions": [
            {
                "question_id": "FAM-Q01",
                "topic": "Household Size and Dependents",
                "prompt_en": "How many individuals live in your household, and how many are young children, elderly, or family members with disabilities?",
                "prompt_taglish": "Ilan po kayong lahat na nakatira sa bahay, at ilan po ang mga bata, senior citizen, o may kapansanan na dumedepende sa inyo?",
                "expected_entities": ["household_size", "number_of_dependents", "vulnerable_members_detail"]
            }
        ]
    },
    "domain_5_psychosocial_circumstances": {
        "title": "Psychosocial Support, Coping & Crisis Dynamics",
        "description": "Explores community support, family strain caused by illness, emotional wellbeing, and urgent crisis risks.",
        "core_questions": [
            {
                "question_id": "PSY-Q01",
                "topic": "Family and Community Support Network",
                "prompt_en": "Who provides emotional or physical care for you at home during your illness, and are relatives or neighbors able to assist?",
                "prompt_taglish": "Sino po ang pangunahing nag-aalaga sa inyo sa araw-araw ngayong malabo ang inyong mata, at may maaasahan po ba kayong tulong sa inyong mga kamag-anak o kapitbahay?",
                "expected_entities": ["primary_caregiver", "community_support_network", "family_stress_factors"]
            }
        ]
    }
}

DOMAIN_PROMPTS_JSON.write_text(json.dumps(domain_prompts, indent=2), encoding='utf-8')
print(f"Exported multi-domain interview prompts -> {DOMAIN_PROMPTS_JSON}")


### 4. Contextual Follow-Up Generation Prompt
When a patient responds with an incomplete, vague, or emotionally heavy statement, the assistant suggests 2-3 focused follow-up questions tailored to the missing detail.


In [ ]:
followup_prompt_template = {
    "template_name": "contextual_msw_followup_generator",
    "system_instructions": "Based on the conversation history and the patient's latest statement, generate 2-3 concise, empathetic follow-up questions for the MSW. Do not repeat facts already confirmed. If an answer mentions an expense without an amount, gently ask for the estimate.",
    "input_format": {
        "domain": "string (e.g., financial_circumstances)",
        "confirmed_facts": "list of key-value pairs already recorded",
        "patient_latest_statement": "verbatim text of patient's response",
        "missing_attributes": "list of attributes required for this domain"
    },
    "output_schema": {
        "type": "object",
        "properties": {
            "missing_information_detected": {"type": "array", "items": {"type": "string"}},
            "suggested_followups": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "question_id": {"type": "string"},
                        "target_entity": {"type": "string"},
                        "suggested_question_taglish": {"type": "string"},
                        "suggested_question_en": {"type": "string"},
                        "clinical_rationale": {"type": "string"}
                    },
                    "required": ["question_id", "target_entity", "suggested_question_taglish", "suggested_question_en"]
                }
            }
        },
        "required": ["missing_information_detected", "suggested_followups"]
    }
}

FOLLOWUP_PROMPTS_JSON.write_text(json.dumps(followup_prompt_template, indent=2), encoding='utf-8')
print(f"Exported follow-up generator template -> {FOLLOWUP_PROMPTS_JSON}")


### Summary of Notebook 02 Deliverables
- [x] Defined and exported the core MSW System Prompt with strict guardrails (`msw_assistant_system_prompt.txt`).
- [x] Implemented multi-domain interview templates covering 5 clinical and psychosocial domains (`domain_interview_prompts.json`).
- [x] Established contextual follow-up generation templates with schema validation (`followup_generation_prompts.json`).
- [x] Verified zero leakage and zero unauthorized decision-making rules in all prompt templates.
